create daily/weekly rss feed for selected rss feeds and summarize 

In [1]:
import feedparser
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from openai import OpenAI
import os

In [2]:
def fetch_rss_entries(feed_urls, days_limit=7):
    entries = []
    cutoff_date = datetime.utcnow() - timedelta(days=days_limit)
    
    for url in feed_urls:
        feed = feedparser.parse(url)
        for entry in feed.entries:
            published = datetime(*entry.published_parsed[:6])
            if published > cutoff_date:
                entries.append({
                    'title': entry.title,
                    'link': entry.link,
                    'published': published,
                    'summary': BeautifulSoup(entry.summary, 'html.parser').text
                })
                # print(entry)
    return sorted(entries, key=lambda x: x['published'], reverse=True)


In [88]:
audience = {
    "general": "general audience with an interest in science, non-specialists, educated laypeople"
}

content_types = {
    "news": "news"
}

In [89]:
def get_system_prompt(subject_area="space", selected_audience="scientists"):
    system_prompt = f"""You are a science writer creating summaries for a blog focussed on {subject_area}. Your summaries should be clear, concise, and engaging, suitable for {audience[selected_audience]}."""
    return system_prompt

In [90]:
def get_user_prompt_news(entry, subject_area="space"):
    user_prompt = f"""
    Summarize the following {subject_area} artcle. Include:
        - The topic being discussed
        - The date this is happening / happened
        - Why is this exciting
        - Any other relevant information

    Avoid copying directly from the abstract unless unavoidable. Keep it sharp and readable.

    Title: {entry['title']}
    Summary: {entry['summary']}
    Link: {entry['link']}
    """
    return user_prompt

In [91]:
def get_user_prompt_newss(entries, subject_area="space", top_entries=5, summary_length=500):
    news_blocks = "\n\n".join(
        f"Title: {e['title']}\nSummary: {e['summary']}\nLink: {e['link']}"
        for e in entries
    )

    user_prompt = f"""
    Based on the newss below, write a clear and engaging summary of **what’s happening today in {subject_area}**. 

    Your task:

    1. **Group the newss into themes** based on their topics 
    2. For each theme, summarize:
        - The topic being discussed
        - The date this is happening / happened
        - Why is this exciting
        - Any other relevant information
    3. End with a section titled "**Top {top_entries} Space News Today**" — list the {top_entries} most interesting or impactful newss with a one-line summary and their links.

    Avoid copying directly from the summaries unless needed and try to keep it under {summary_length}

    Here are the newss to consider:

    {news_blocks}
    """.strip()

    return user_prompt

In [92]:
def create_message_entry(entry, subject_area, selected_audience, content_type="news"):
    system_prompt = get_system_prompt(subject_area, selected_audience)
    user_prompt = get_user_prompt_news(entry, subject_area)
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

def create_message_entries(entries, subject_area, selected_audience, content_type="news"):
    system_prompt = get_system_prompt(subject_area, selected_audience)
    user_prompt = get_user_prompt_newss(entries, subject_area, top_entries=5, summary_length=500)
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [93]:
def call_openai(messages, model="gpt-4", temperature=0.5):
    openai_api_key = os.getenv('OPENAI_API_KEY')
    openai = OpenAI(api_key=openai_api_key)
    response = openai.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
        )
    return response.choices[0].message.content


In [ ]:
SPACE_NEWS_FEEDS = [
    "https://news.mit.edu/topic/mitspace-rss.xml",
    "https://www.nasa.gov/news-release/feed/",
    "https://www.esa.int/rssfeed/Our_Activities/Space_Science",
]

In [ ]:
entries = fetch_rss_entries(SPACE_NEWS_FEEDS, days_limit=15)

In [95]:
print(len(entries))

0


In [96]:
selected_audience = "general"
subject_area = "space"
content_type = "news"
messages = create_message_entries(entries, subject_area, selected_audience, content_type)
# messages

In [35]:
messages

[{'role': 'system',
  'content': 'You are a science writer creating summaries for a blog focussed on space. Your summaries should be clear, concise, and engaging, suitable for general audience with an interest in science, non-specialists, educated laypeople.'},
 {'role': 'user',
  'content': 'Based on the newss below, write a clear and engaging summary of **what’s happening today in space**. \n\n    Your task:\n\n    1. **Group the newss into themes** based on their topics \n    2. For each theme, summarize:\n        - The topic being discussed\n        - The date this is happening / happened\n        - Why is this exciting\n        - Any other relevant information\n    3. End with a section titled "**Top 5 Space News Today**" — list the 5 most interesting or impactful newss with a one-line summary and their links.\n\n    Avoid copying directly from the summaries unless needed and try to keep it under 500\n\n    Here are the newss to consider:\n\n    Title: Completed Plato spacecraft i

In [36]:
print(call_openai(messages, model="gpt-4", temperature=0.5))

**Theme 1: Mars Exploration**

- **Topic**: Observation of Mars' atmospheric conditions and topography.
- **Date**: Various dates.
- **Exciting Because**: These observations provide a deeper understanding of Mars' geological processes and atmospheric conditions, which could aid future manned missions to the planet.
- **Relevant Information**: The European Space Agency's (ESA) Mars Express spacecraft has been observing dust devils on Mars, providing insight into the planet's wind patterns. Additionally, the Mars Express has also flown over Xanthe Terra, capturing stunning imagery of the Martian terrain. NASA and Blue Origin are also preparing for the launch of the ESCAPADE mission, which will study the interaction between the solar wind and Mars' atmosphere.

**Theme 2: Spacecraft and Mission Preparations**

- **Topic**: Preparation and testing of spacecraft for various space missions.
- **Date**: Various dates.
- **Exciting Because**: These missions aim to explore and understand more a

In [37]:
for entry in entries:
    print(f"🔹 {entry['title']} ({entry['published'].strftime('%Y-%m-%d')})")
    print(f"🔗 {entry['link']}")
    messages = create_message_entry(entry, subject_area, selected_audience, content_type)
    print("🧠 Summary:", call_openai(messages, model="gpt-4", temperature=0.5))
    print("\n---\n")


🔹 Completed Plato spacecraft is ready for final tests (2025-10-09)
🔗 https://www.esa.int/Science_Exploration/Space_Science/Plato/Completed_Plato_spacecraft_is_ready_for_final_tests
🧠 Summary: Title: Plato Spacecraft Nears Completion, Ready for Final Tests

The European Space Agency (ESA) has announced the completion of the construction of Plato, a spacecraft designed to discover Earth-like exoplanets. The spacecraft, now fitted with its sunshield and solar panels, is ready for the final tests before its much-anticipated launch. 

The completion of the Plato spacecraft is a remarkable milestone in space exploration. This news is particularly exciting for astrobiology enthusiasts and scientists alike, as Plato's mission is to seek out planets beyond our solar system that could potentially harbor life. 

The date of the announcement was not specified in the article, but the spacecraft is now moving into the final testing phase. This will ensure Plato is fully operational and ready to emba

In [ ]:
# "https://www.deeplearning.ai/thebatch/feed.xml",
#  "https://jack-clark.net/feed/",
#   "https://thegradient.pub/rss/",
#  "https://paperswithcode.com/latest/rss",
#  "https://machinelearningmastery.com/blog/feed/"

In [ ]:
SELECTED_NEWS_FEEDS = [
    # "https://medium.com/feed/@Pinterest_Engineering",
    # "https://medium.com/feed/netflix-techblog",
    # "https://medium.com/feed/airbnb-engineering",

    # "https://distill.pub/rss.xml",
    # "https://news.mit.edu/topic/mitartificial-intelligence2-rss.xml",
    # "https://bair.berkeley.edu/blog/feed.xml",
    # "https://vectorinstitute.ai/feed/",
    # "https://aws.amazon.com/blogs/ai/feed",
    # "https://research.google/blog/rss",
    # "https://www.deepmind.com/blog/rss.xml",
    # "https://openai.com/news/rss.xml",
    # "https://www.microsoft.com/en-us/research/feed/",
    # "https://developer.nvidia.com/blog/feed/",
    # "https://spectrum.ieee.org/feeds/topic/artificial-intelligence.rss",

    # "https://rss.arxiv.org/rss/cs.AI",
    # "https://rss.arxiv.org/rss/stat.ML",
    # "https://rss.arxiv.org/rss/cs.LG",
    # "https://rss.arxiv.org/rss/cs.CV",  
    # "https://rss.arxiv.org/rss/cs.CL"
    # "https://rss.arxiv.org/rss/cs.GR",
    # "https://rss.arxiv.org/rss/cs.GT",
    # "https://rss.arxiv.org/rss/cs.CG",
    # "https://rss.arxiv.org/rss/cs.NE",

    # "https://allenai.org/papers",# "https://huggingface.co/blog/feed.xml"# "https://huggingface.co/papers/feed.xml" #"https://eng.uber.com/category/articles/ai/feed"

    # "https://www.sciencedaily.com/rss/space_time/astrophysics.xml"

    # "https://phys.org/rss-feed/space-news",

    # "https://www.esa.int/rssfeed/TopNews",
    # "https://www.esa.int/rssfeed/Our_Activities/Space_Science",

    # "https://www.nasa.gov/news-release/feed",  # OR "https://www.nasa.gov/feed/",
    # "https://www.nasa.gov/technology/feed/",
    # "https://www.nasa.gov/aeronautics/feed/",
    # "https://www.nasa.gov/missions/station/feed/",
    # "https://www.nasa.gov/missions/artemis/feed/",
    
    # "https://www.space.com/feeds/all",
    # "https://www.space.com/feeds/tag/space-exploration",
    # "https://www.space.com/feeds/tag/space-science",
    # "https://www.space.com/feeds/tag/astronomy",
    # "https://www.space.com/feeds/tag/universe",
    # "https://www.space.com/feeds/tag/stargazing"

    # "https://skyandtelescope.org/rss",

    # "https://news.mit.edu/topic/mitspace-rss.xml",
    # "https://news.mit.edu/topic/mitastrophysics-rss.xml",
    # "https://news.mit.edu/topic/mitspace-exploration-rss.xml",

    # "http://export.arxiv.org/rss/astro-ph"
    # "https://rss.arxiv.org/rss/astro-ph.CO",
    # "https://rss.arxiv.org/rss/astro-ph.GA",
    # "https://rss.arxiv.org/rss/astro-ph.HE",
    # "https://rss.arxiv.org/rss/astro-ph.IM",
    # "https://rss.arxiv.org/rss/astro-ph.SR",
    # "https://rss.arxiv.org/rss/astro-ph.EP"


    # "https://keckobservatory.org/feed/",
    # "https://news.harvard.edu/gazette/section/science-technology/feed/"
    
    # "https://www.sdss3.org/press/",
    # "https://news.stanford.edu/artificial-intelligence/",
    # "https://kipac.stanford.edu/news-categories/news",
    # "https://www.vlbi.at/news/"

]
entries = fetch_rss_entries(SELECTED_NEWS_FEEDS, days_limit=30)
print(len(entries))

0
